# Static large deviations of the subvolume density in 2D active Brownian particles

The large parameter is the **subvolume area** $v=\ell^2$, not time:

$$P_v(N_v=\rho v)\asymp e^{-v I(\rho)},\qquad \psi_v(\lambda)=\frac1v\ln\langle e^{\lambda N_v}\rangle_{ss}\to\psi(\lambda),\qquad I(\rho)=\sup_\lambda[\lambda\rho-\psi(\lambda)].$$

The bias (`biasing` $=\lambda$) acts on the instantaneous particle number $N_v$ in a central $\ell\times\ell$ square of an $L\times L$ periodic box with $L=\kappa\ell$.
It is sampled exactly, whatever the (unknown, non-Boltzmann) steady state is, by finite-horizon, Doob-guided, twisted Feynman–Kac SMC started from steady-state configurations (see `README.md`).

This notebook (1) runs a small demonstration and (2) loads the production results written by `scripts/run_subvolume_ldp.py` and `scripts/analyze_subvolume_ldp.py`.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from abp_ldp.simulator import ABPParams, Geometry
from abp_ldp.smc import SMCConfig, ReservoirConfig, Reservoir, run_lambda_grid, combine_replicates
from abp_ldp import analysis as an

## 1. Small demonstration (one subvolume size, a few biasing fields)

A few minutes on a laptop. Production settings: `M=128`, `R=3`, horizon $2.5\,\tau_v$, $\ell\in\{6,\dots,14\}$.

In [ ]:
params = ABPParams(rho0=0.4, v0=24.0, Dr=3.0, dt=1e-4)          # Pe = 24, WCA
geom = Geometry(ell=6.0, kappa=3.0)                               # v = 36, L = 18
M, R = 64, 3
res = Reservoir(params, geom, ReservoirConfig(M=M * R, t_burn=3.0, t_prod=1.0, seed=1)).build()
v_eff, Dt_eff, _ = res.measure_self_propagation()                 # effective one-body twist parameters
horizon = 2.5 * res.tau_v
lambdas = [-1.0, -0.5, 0.5, 1.0]
out = run_lambda_grid(params, geom, SMCConfig(M=M, seed=2), res, lambdas, R, horizon, v_eff, Dt_eff)
lnZ, err = combine_replicates(out["lnZ_reps"])
for l, z, e in zip(lambdas, lnZ, err):
    print(f"lambda={l:+.2f}   psi_v = lnZ/v = {z / geom.v:+.4f} +- {e / geom.v:.4f}")

In [ ]:
# brute-force check near the peak: ln <exp(lambda N_v)> from unbiased histograms (all tiled placements)
from scipy.special import logsumexp
lnP_bf = res.brute_force_lnP()
N = np.arange(lnP_bf.size); ok = np.isfinite(lnP_bf)
for l, z in zip(lambdas, lnZ):
    print(f"lambda={l:+.2f}   SMC lnZ = {z:8.3f}   brute force = {logsumexp(lnP_bf[ok] + l * N[ok]):8.3f}")

## 2. Production results

```bash
python scripts/run_subvolume_ldp.py --out results/abp_pe24 --ells 6 8 10 12 14
python scripts/run_subvolume_ldp.py --preset ideal --out results/ideal --ells 6 8 10 12 14
python scripts/analyze_subvolume_ldp.py results/abp_pe24 --ideal results/ideal
```

In [ ]:
from IPython.display import Image, Markdown, display
for f in ["fig_volume_scaling", "fig_scgf", "fig_rate_function", "fig_validation", "fig_diagnostics"]:
    display(Image(f"results/abp_pe24/{f}.png"))
display(Markdown(open("results/abp_pe24/summary.md").read()))